# Lighthouse — turn classifier fine-tune (Colab T4)

Fine-tunes the DistilBERT turn-level harm classifier off the local machine. The M2 is a
flat ~2.2k tokens/s wall for this model (`docs/context.md` §9), which put the 3-epoch run
at ~2.3 hours and made the laptop unusable. A free T4 does the same run in roughly
10-15 minutes.

**The training code is not duplicated here.** This notebook uploads `ml/lighthouse/` and
runs `python -m lighthouse.model.train_turn` unchanged, so the numbers it produces are
the numbers the repo's script produces. Do not edit hyperparameters in this notebook —
edit `ml/lighthouse/config.py`, rebuild the bundle, re-upload.

## Before you start

1. Locally: `bash ml/colab/make_bundle.sh` → writes `data/artifacts/lighthouse_bundle.tar.gz`
2. Here: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**
3. Cell 3 will prompt you to pick the bundle file

Total hands-on time is the upload and the two downloads at the end.

## 1. Confirm the GPU

This cell **fails loudly if there is no GPU**, on purpose. A silent fallback to Colab's
CPU would not error, it would just run for hours and look like it was working — which is
the exact failure mode that cost us two sessions on the laptop.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun. "
    "Training on Colab's CPU takes hours and is not worth doing."
)
print(f"torch {torch.__version__}")
print(f"device: {torch.cuda.get_device_name(0)}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Dependencies

Colab ships torch with CUDA, plus numpy / pandas / scikit-learn / pyarrow. Only
`transformers` and `accelerate` need installing. Torch is deliberately left alone:
reinstalling it here is the fastest way to end up with a CPU-only build.

In [ ]:
!pip install -q -U transformers accelerate

import transformers, sklearn, pandas as pd, numpy as np
print(f"transformers {transformers.__version__} | sklearn {sklearn.__version__} | "
      f"pandas {pd.__version__} | numpy {np.__version__}")

## 3. Upload the bundle

Pick `lighthouse_bundle.tar.gz` (~16MB) when the file dialog opens. It carries the
`lighthouse` package and the three split parquets at the same relative paths the repo
uses, so `config.py` resolves `data/splits/` with no env vars and no path edits.

In [ ]:
import pathlib, shutil, tarfile
from google.colab import files

ROOT = pathlib.Path("/content/lighthouse")
if ROOT.exists():
    shutil.rmtree(ROOT)   # a re-run must not train against a half-overwritten old bundle
ROOT.mkdir(parents=True)

uploaded = files.upload()
name = next(iter(uploaded))
with tarfile.open(name) as tar:
    tar.extractall(ROOT)

for split in ("train", "val", "test"):
    p = ROOT / "data" / "splits" / f"turns_{split}.parquet"
    assert p.exists(), f"bundle is missing {p.name} — rebuild it with make_bundle.sh"
    print(f"{p.name:24} {p.stat().st_size / 1e6:6.1f} MB")

## 4. Sanity check the corpus

Cheap, and it catches an upload that silently truncated. The class counts must match
`docs/context.md` §10: train 45,286 rows, `threat` the scarce class at 327.

In [ ]:
%cd /content/lighthouse/ml
import pandas as pd

for split in ("train", "val", "test"):
    df = pd.read_parquet(f"../data/splits/turns_{split}.parquet")
    print(f"\n{split}  {len(df):,} rows")
    print(df["harm"].value_counts().to_string())

## 5. Train

Runs the repo's script unchanged: 3 epochs, class-weighted cross-entropy,
length-grouped batching, best-epoch-by-val-macro-F1 checkpointing.

**What to read in the output, in order** (this ordering is deliberate, see the module
docstring in `train_turn.py`):

1. **`pad-eff`** in the step lines — should sit near 0.97. If it collapses to ~0.36 the
   length-grouped sampler is not doing its job and the run is paying for 2.5x the tokens.
2. **The DISTRESS/SELF_HARM confusion pair** — the hardest and most consequential
   boundary in the taxonomy. Day 1's TF-IDF baseline confused these 736 times.
3. **The safety view** — risky turns missed entirely as benign.
4. **Per-class F1**, especially THREAT (70 test examples, moves in jumps).
5. **Macro-F1** last, against the 0.7148 baseline.

Expect roughly 10-15 minutes on a T4. Leave the tab focused; Colab reclaims idle
sessions.

In [ ]:
%cd /content/lighthouse/ml
# -u is required or the step counter buffers and progress looks frozen.
!python -u -m lighthouse.model.train_turn 2>&1 | tee /content/train_turn.log

## 6. Download the small artifacts — do this one first

`turn_logits.npz` (raw val/test logits) and `turn_distilbert.json` (all the metrics)
together are under a megabyte, and they are what unblocks the rest of day 2: temperature
scaling and the reliability diagram both read the logits, and neither needs the
checkpoint. The training log rides along so the run is reproducible in `docs/log.md`.

Grab these **before** the 265MB checkpoint. If the big download flakes, day 2 is still
finished.

In [ ]:
import shutil, pathlib
from google.colab import files

ART = pathlib.Path("/content/lighthouse/data/artifacts")
shutil.copy("/content/train_turn.log", ART / "train_turn.log")

small = pathlib.Path("/content/lighthouse_metrics.zip")
!cd {ART} && zip -q -r {small} turn_logits.npz turn_distilbert.json train_turn.log
print(f"{small.stat().st_size / 1e6:.2f} MB")
files.download(str(small))

## 7. Download the checkpoint

~265MB of DistilBERT weights plus the tokenizer. Needed for the day 9 HF Space and any
live inference, **not** needed for today's calibration work.

If this download stalls (Colab's `files.download` is unreliable at this size), run the
Drive fallback cell below instead.

In [ ]:
import pathlib
from google.colab import files

ckpt = pathlib.Path("/content/lighthouse_turn_model.zip")
!cd /content/lighthouse/data/artifacts && zip -q -r {ckpt} turn_model
print(f"{ckpt.stat().st_size / 1e6:.0f} MB")
files.download(str(ckpt))

### Fallback: copy to Google Drive

Only needed if the direct download above failed. Copies both zips to
`MyDrive/lighthouse/`, which you then download from Drive at your leisure.

In [ ]:
import pathlib, shutil
from google.colab import drive

drive.mount("/content/drive")
dest = pathlib.Path("/content/drive/MyDrive/lighthouse")
dest.mkdir(parents=True, exist_ok=True)
for z in ("/content/lighthouse_metrics.zip", "/content/lighthouse_turn_model.zip"):
    shutil.copy(z, dest)
    print(f"copied {z} -> {dest}")

## Back on the laptop

```bash
cd <repo>
unzip -o ~/Downloads/lighthouse_metrics.zip    -d data/artifacts/
unzip -o ~/Downloads/lighthouse_turn_model.zip -d data/artifacts/

cd ml && .venv/bin/python -m lighthouse.model.calibrate_turn
```

Calibration is a single-parameter LBFGS fit on CPU: seconds, not hours. It writes the
before/after reliability diagram to `data/artifacts/reliability_turn.png`, which is the
last open item on day 2.

Everything under `data/` is gitignored, so none of these downloads can be committed by
accident.